# Modul 10: Regressions- und Klassifikationsmodelle vergleichen

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Regression vergleichen, Klassifikation vergleichen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 120 bis 170 Minuten

    ## Überblick

    Sie vergleichen lineare, robuste, nachbarschaftsbasierte, baumbasierte und kernelbasierte Modelle auf identischen Splits. Bewertung, Residuen, Lernkurven, Wahrscheinlichkeiten, Gewichte, Voting und Fehleranalyse stehen im Mittelpunkt.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_10A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_10B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Lineare, regularisierte und robuste Regressionsmodelle in Pipelines trainieren.
- Nichtlineare Regressoren wie k-NN, Bäume, Forest, Boosting und SVR vergleichen.
- Regression mit MAE, RMSE, R², Lernkurven und Laufzeit bewerten.
- Lineare, probabilistische, nachbarschafts- und baumbasierte Klassifikatoren trainieren.
- Wahrscheinlichkeiten, Klassen- und Stichprobengewichte sowie Voting untersuchen.
- Klassifikatoren mit Konfusionsmatrizen und Fehleranalysen vergleichen.

    ## Bewertete Fähigkeiten

    - Pipeline-basierter Modellvergleich auf identischen Daten
- MAE, RMSE, R², Residuen, Laufzeit und Lernkurven
- LogReg, Naive Bayes, k-NN, Baum, Forest, Boosting und SVC
- Klassenunwucht, class_weight, Voting und Fehlklassifikationen

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
from sklearn.datasets import make_classification, make_friedman1
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.linear_model import HuberRegressor, LinearRegression, LogisticRegression, Ridge, SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_regression, y_regression = make_friedman1(
    n_samples=500,
    n_features=7,
    noise=1.2,
    random_state=RANDOM_SEED,
)
X_classification_10, y_classification_10 = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.72, 0.28],
    class_sep=1.0,
    flip_y=0.04,
    random_state=RANDOM_SEED,
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Lineare, regularisierte und robuste Regression vergleichen

    Erstellen Sie einen festen Train/Test-Split für `X_regression`, `y_regression`. Vergleichen Sie:

- `LinearRegression`,
- `Ridge(alpha=1.0)` in einer Skalierungspipeline,
- `HuberRegressor` in einer Skalierungspipeline.

Messen Sie Fit-Zeit, MAE, RMSE und R². Erstellen Sie für jedes Modell ein eigenes Residuenplot.

> **Hinweis:** Verwenden Sie für alle Modelle denselben Split und dieselben Testziele.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_regression, y_regression, test_size=0.25, random_state=RANDOM_SEED
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Lineare, regularisierte und robuste Regression vergleichen
#
# Ziel dieser Codezelle:
# Erstellen Sie einen festen Train/Test-Split für Xregression, yregression.
# Vergleichen Sie: - LinearRegression, - Ridge(alpha=1.0) in einer
# Skalierungspipeline, - HuberRegressor in einer Skalierungspipeline. Messen Sie...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_regression,
    y_regression,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

regression_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Huber": make_pipeline(StandardScaler(), HuberRegressor(max_iter=500)),
}

regression_rows = []
regression_predictions = {}

for model_name, model in regression_models.items():
    start_time = time.perf_counter()
    model.fit(X_train_r, y_train_r)
    fit_seconds = time.perf_counter() - start_time

    predictions = model.predict(X_test_r)
    regression_predictions[model_name] = predictions

    mse = mean_squared_error(y_test_r, predictions)
    regression_rows.append(
        {
            "model": model_name,
            "fit_seconds": fit_seconds,
            "MAE": mean_absolute_error(y_test_r, predictions),
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y_test_r, predictions),
        }
    )

    residuals = y_test_r - predictions
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(predictions, residuals, alpha=0.7)
    ax.axhline(0, linewidth=1)
    ax.set_title(f"Residuen: {model_name}")
    ax.set_xlabel("Vorhersage")
    ax.set_ylabel("Residuum")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

linear_regression_comparison = (
    pd.DataFrame(regression_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
print(linear_regression_comparison.round(4).to_string(index=False))

### Reflexion zu Aufgabe 1

Friedman1 enthält nichtlineare Beziehungen, daher können rein lineare Modelle systematische Residuen zeigen. Ridge begrenzt Koeffizienten, ist aber weiterhin linear. Huber reduziert den Einfluss großer Residuen. Ein Modell sollte nicht allein nach R² gewählt werden; Fehlergröße, Muster, Laufzeit und Einsatzanforderungen gehören gemeinsam betrachtet.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Nichtlineare Regressoren fair vergleichen

    Vergleichen Sie auf demselben Split:

- k-NN-Regression in einer Skalierungspipeline,
- Entscheidungsbaum,
- Random Forest,
- Gradient Boosting,
- SVR in einer Skalierungspipeline.

Begrenzen Sie Modellgrößen sinnvoll. Erstellen Sie eine gemeinsame Ergebnistabelle aus Aufgabe 1 und 2 und zeigen Sie tatsächliche gegen vorhergesagte Werte für das beste Modell.

> **Hinweis:** Begrenzen Sie Baumtiefe und Anzahl der Bäume, damit der Vergleich Colab-freundlich bleibt.

In [ ]:
# Führen Sie Aufgabe 1 zuerst aus, damit der gemeinsame Split verfügbar ist.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Nichtlineare Regressoren fair vergleichen
#
# Ziel dieser Codezelle:
# Vergleichen Sie auf demselben Split: - k-NN-Regression in einer
# Skalierungspipeline, - Entscheidungsbaum, - Random Forest, - Gradient Boosting, -
# SVR in einer Skalierungspipeline. Begrenzen Sie Modellgrößen sinnvoll....
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

nonlinear_models = {
    "k-NN": make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=8),
    ),
    "Entscheidungsbaum": DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=5,
        random_state=RANDOM_SEED,
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_SEED,
    ),
    "SVR": make_pipeline(
        StandardScaler(),
        SVR(C=10.0, epsilon=0.15, gamma="scale"),
    ),
}

nonlinear_rows = []
nonlinear_predictions = {}

for model_name, model in nonlinear_models.items():
    start_time = time.perf_counter()
    model.fit(X_train_r, y_train_r)
    fit_seconds = time.perf_counter() - start_time
    predictions = model.predict(X_test_r)
    nonlinear_predictions[model_name] = predictions

    mse = mean_squared_error(y_test_r, predictions)
    nonlinear_rows.append(
        {
            "model": model_name,
            "fit_seconds": fit_seconds,
            "MAE": mean_absolute_error(y_test_r, predictions),
            "RMSE": np.sqrt(mse),
            "R2": r2_score(y_test_r, predictions),
        }
    )

all_regression_results = (
    pd.concat(
        [
            linear_regression_comparison,
            pd.DataFrame(nonlinear_rows),
        ],
        ignore_index=True,
    )
    .sort_values("RMSE")
    .reset_index(drop=True)
)

best_regression_name = all_regression_results.loc[0, "model"]
if best_regression_name in nonlinear_predictions:
    best_regression_predictions = nonlinear_predictions[best_regression_name]
else:
    best_regression_predictions = regression_predictions[best_regression_name]

print(all_regression_results.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_r, best_regression_predictions, alpha=0.7)
limits = [
    min(y_test_r.min(), best_regression_predictions.min()),
    max(y_test_r.max(), best_regression_predictions.max()),
]
ax.plot(limits, limits, linestyle="--")
ax.set_title(f"Tatsächlich gegen vorhergesagt: {best_regression_name}")
ax.set_xlabel("Tatsächlicher Wert")
ax.set_ylabel("Vorhersage")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 2

Nichtlineare Modelle können die Struktur von Friedman1 besser erfassen. k-NN und SVR benötigen Skalierung, Bäume dagegen nicht. Random Forest und Boosting kombinieren mehrere Bäume, unterscheiden sich aber im Lernprinzip. Die gemessene Laufzeit ist hardwareabhängig und in kleinen Datensätzen nur eine grobe Orientierung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Manuelle Lernkurve für das beste Regressionsmodell

    Verwenden Sie das beste Regressionsmodell aus Aufgabe 2. Trainieren Sie frische Modellinstanzen mit 10, 25, 50, 75 und 100 Prozent der Trainingsdaten. Berechnen Sie jeweils Trainings- und Test-RMSE.

Zeichnen Sie beide Kurven und interpretieren Sie, ob mehr Daten wahrscheinlich helfen und ob Anzeichen für Unter- oder Überanpassung bestehen.

> **Hinweis:** Jede Kurvenstelle braucht ein neu trainiertes Modell.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Manuelle Lernkurve für das beste Regressionsmodell
#
# Ziel dieser Codezelle:
# Verwenden Sie das beste Regressionsmodell aus Aufgabe 2. Trainieren Sie frische
# Modellinstanzen mit 10, 25, 50, 75 und 100 Prozent der Trainingsdaten. Berechnen
# Sie jeweils Trainings- und Test-RMSE. Zeichnen Sie beide...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Für einen reproduzierbaren Lernkurvenvergleich wird hier das gut
# geeignete Gradient-Boosting-Modell als feste Kandidatenklasse gewählt.
train_fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
learning_curve_rows = []

permutation = np.random.default_rng(RANDOM_SEED).permutation(len(X_train_r))
X_train_shuffled = X_train_r[permutation]
y_train_shuffled = y_train_r[permutation]

for fraction in train_fractions:
    subset_size = max(20, int(fraction * len(X_train_shuffled)))
    X_subset = X_train_shuffled[:subset_size]
    y_subset = y_train_shuffled[:subset_size]

    # Für jede Größe wird eine neue Instanz trainiert. Ein bereits
    # gefittetes Modell darf nicht weiterverwendet werden.
    model = GradientBoostingRegressor(
        n_estimators=120,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_SEED,
    )
    model.fit(X_subset, y_subset)

    train_predictions = model.predict(X_subset)
    test_predictions = model.predict(X_test_r)

    learning_curve_rows.append(
        {
            "train_size": subset_size,
            "train_RMSE": np.sqrt(
                mean_squared_error(y_subset, train_predictions)
            ),
            "test_RMSE": np.sqrt(
                mean_squared_error(y_test_r, test_predictions)
            ),
        }
    )

learning_curve = pd.DataFrame(learning_curve_rows)
print(learning_curve.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(learning_curve["train_size"], learning_curve["train_RMSE"], marker="o", label="Training")
ax.plot(learning_curve["train_size"], learning_curve["test_RMSE"], marker="o", label="Test")
ax.set_title("Manuelle Lernkurve")
ax.set_xlabel("Anzahl Trainingsbeobachtungen")
ax.set_ylabel("RMSE")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 3

Bei kleinen Trainingsmengen kann der Trainingsfehler sehr niedrig und der Testfehler deutlich höher sein. Das ist ein Hinweis auf hohe Varianz beziehungsweise Überanpassung. Wenn sich der Testfehler mit mehr Daten weiter verbessert, könnten zusätzliche repräsentative Daten helfen. Bleiben beide Fehler hoch und ähnlich, wäre eher Modellform oder Merkmalsdarstellung zu überdenken.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Klassifikatoren, Wahrscheinlichkeiten und Konfusionsmatrizen vergleichen

    Erstellen Sie einen stratifizierten Split für `X_classification_10`, `y_classification_10`. Vergleichen Sie mindestens:

- logistische Regression,
- SGD-Klassifikator mit Log-Loss,
- Gaussian Naive Bayes,
- k-NN,
- Entscheidungsbaum,
- Random Forest,
- Gradient Boosting,
- SVC mit Wahrscheinlichkeiten.

Verwenden Sie geeignete Skalierungspipelines. Berechnen Sie Accuracy und die Konfusionsmatrix jedes Modells. Speichern Sie außerdem die positive Klassenwahrscheinlichkeit, soweit verfügbar.

> **Hinweis:** Vergleichen Sie besonders FN und FP, nicht nur die Gesamtsumme richtiger Labels.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_classification_10, y_classification_10, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_classification_10
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Klassifikatoren, Wahrscheinlichkeiten und Konfusionsmatrizen vergleichen
#
# Ziel dieser Codezelle:
# Erstellen Sie einen stratifizierten Split für Xclassification10,
# yclassification10. Vergleichen Sie mindestens: - logistische Regression, - SGD-
# Klassifikator mit Log-Loss, - Gaussian Naive Bayes, - k-NN, - Entscheidun...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_classification_10,
    y_classification_10,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_classification_10,
)

classifiers = {
    "LogReg": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    ),
    "SGD": make_pipeline(
        StandardScaler(),
        SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-4,
            random_state=RANDOM_SEED,
        ),
    ),
    "GaussianNB": GaussianNB(),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "Baum": DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=RANDOM_SEED),
    "RandomForest": RandomForestClassifier(n_estimators=120, max_depth=8, min_samples_leaf=3, random_state=RANDOM_SEED, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=RANDOM_SEED),
    "SVC": make_pipeline(StandardScaler(), SVC(C=2.0, probability=True, random_state=RANDOM_SEED)),
}

classification_rows = []
classifier_predictions = {}
classifier_probabilities = {}

for model_name, model in classifiers.items():
    start_time = time.perf_counter()
    model.fit(X_train_c, y_train_c)
    fit_seconds = time.perf_counter() - start_time

    predictions = model.predict(X_test_c)
    classifier_predictions[model_name] = predictions

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_test_c)[:, 1]
        classifier_probabilities[model_name] = probabilities

    matrix = confusion_matrix(y_test_c, predictions)
    tn, fp, fn, tp = matrix.ravel()
    classification_rows.append(
        {
            "model": model_name,
            "fit_seconds": fit_seconds,
            "accuracy": accuracy_score(y_test_c, predictions),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

classification_comparison = (
    pd.DataFrame(classification_rows)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)
print(classification_comparison.round(4).to_string(index=False))

### Reflexion zu Aufgabe 4

Accuracy allein kann bei der ungleichen Klassenverteilung wichtige Fehler verbergen. Die Konfusionsmatrix zeigt getrennt, wie viele positive Fälle übersehen und wie viele negative Fälle fälschlich markiert wurden. Modelle ohne sinnvoll interpretierbare Wahrscheinlichkeiten sollten nicht so behandelt werden, als wären ihre Scores automatisch kalibriert.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Klassenwichte, Voting und Fehleranalyse

    Verwenden Sie den Klassifikationssplit aus Aufgabe 4.

1. Vergleichen Sie logistische Regression mit und ohne `class_weight="balanced"`.
2. Bauen Sie einen Soft-Voting-Klassifikator aus logistischer Regression, Random Forest und SVC.
3. Vergleichen Sie Accuracy, Precision und Recall für die positive Klasse.
4. Erstellen Sie eine Tabelle aller Testfälle, bei denen das Voting-Modell falsch liegt, einschließlich Wahrscheinlichkeit und wahrer Klasse.
5. Diskutieren Sie, ob eine höhere positive Recall-Rate die zusätzlichen falsch-positiven Fälle rechtfertigt.

> **Hinweis:** Bewerten Sie Klassenwichte und Schwellenwerte mit fachlichen Fehlerkosten.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Klassenwichte, Voting und Fehleranalyse
#
# Ziel dieser Codezelle:
# Verwenden Sie den Klassifikationssplit aus Aufgabe 4. 1. Vergleichen Sie
# logistische Regression mit und ohne classweight="balanced". 2. Bauen Sie einen
# Soft-Voting-Klassifikator aus logistischer Regression, Random For...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def precision_recall_from_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float]:
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return float(precision), float(recall)

unweighted_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
)
balanced_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
)

voting_classifier = VotingClassifier(
    estimators=[
        (
            "logreg",
            make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
            ),
        ),
        (
            "forest",
            RandomForestClassifier(
                n_estimators=120,
                max_depth=8,
                min_samples_leaf=3,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
        ),
        (
            "svc",
            make_pipeline(
                StandardScaler(),
                SVC(C=2.0, probability=True, random_state=RANDOM_SEED),
            ),
        ),
    ],
    voting="soft",
)

evaluated_models = {
    "LogReg": unweighted_logreg,
    "LogReg balanced": balanced_logreg,
    "Soft Voting": voting_classifier,
}

rows = []
for model_name, model in evaluated_models.items():
    model.fit(X_train_c, y_train_c)
    predictions = model.predict(X_test_c)
    precision, recall = precision_recall_from_predictions(y_test_c, predictions)
    rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_test_c, predictions),
            "precision_positive": precision,
            "recall_positive": recall,
        }
    )

weighted_voting_comparison = pd.DataFrame(rows)

voting_predictions = voting_classifier.predict(X_test_c)
voting_probabilities = voting_classifier.predict_proba(X_test_c)[:, 1]
error_mask = voting_predictions != y_test_c
error_table = pd.DataFrame(X_test_c[error_mask])
error_table["true_label"] = y_test_c[error_mask]
error_table["predicted_label"] = voting_predictions[error_mask]
error_table["positive_probability"] = voting_probabilities[error_mask]

print(weighted_voting_comparison.round(3).to_string(index=False))
print("\nFehler des Voting-Modells:")
print(error_table.round(3).to_string(index=False))

### Reflexion zu Aufgabe 5

Klassenwichte verschieben den Optimierungsfokus und können den Recall der selteneren positiven Klasse erhöhen, häufig auf Kosten zusätzlicher falsch-positiver Fälle. Soft Voting mittelt Wahrscheinlichkeiten unterschiedlicher Modellfamilien und kann robuste Entscheidungen liefern, garantiert aber keine Verbesserung. Die angemessene Balance hängt von den realen Folgen beider Fehlerarten ab.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.